In [ ]:
import os
import sys
import torch
from torch.utils.data import DataLoader
from pathlib import Path

project_root = Path(os.getcwd()).parent
print(f"[*] Project root: {project_root}")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.seed import set_seed
set_seed(42)

from src.data.preprocessing.pipeline import Pipeline
from src.data.datasets.universal_dataset import CVADataset

from src.transformer.network import NARCVGenerator 
from src.train.trainer_transformer import CSTrainer

In [ ]:
EPOCHS = 25
BATCH_SIZE = 64
LR = 0.0004
WEIGHT_DECAY = 0.025

TEST_INHIBITOR = "2-mercaptobenzimidazole"
NUM_CYCLE = [1, 2, 3, 4]
SAVE_DIR = str(project_root / "experiments" / "run_nar_transformer")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEQ_LEN = 968         
PATCH_SIZE = 8
D_MODEL = 64
N_HEAD = 4
D_INNER = 32
NUM_LAYERS = 4
NUM_COND_TOKENS = 8
DROPOUT = 0.3

In [3]:
pipe = Pipeline(
    num_cycle=NUM_CYCLE,
    test_inhibitor=TEST_INHIBITOR,
    norm_feat=True,
    use_wavelet=False, 
    flip_the_peak=False
)


train_dataset = CVADataset(
    vol=pipe.train_voltage,
    cur=pipe.train_current,
    desc_df=pipe.train_analyzed_data
)

val_dataset = CVADataset(
    vol=pipe.test_voltage,
    cur=pipe.test_current,
    desc_df=pipe.test_analyzed_data
)

In [4]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Size Train: {len(train_dataset)} samples")
print(f"Size Val: {len(val_dataset)} samples")

num_desc_features = train_dataset[0]["features"].shape[0]

Size Train: 2684 samples
Size Val: 776 samples


In [5]:
model = NARCVGenerator(
    desc_dim=num_desc_features,
    seq_len=SEQ_LEN,
    patch_size=PATCH_SIZE,
    d_model=D_MODEL,
    n_head=N_HEAD,
    d_inner=D_INNER,
    num_layers=NUM_LAYERS,
    num_cond_tokens=NUM_COND_TOKENS,
    dropout=DROPOUT
).to(DEVICE)

In [6]:
trainer = CSTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    save_dir=SAVE_DIR,
    vol_scaler=getattr(pipe, 'vol_scaler', None),
    cur_scaler=getattr(pipe, 'cur_scaler', None),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    epochs=EPOCHS, 
    use_smoothing=False
)

trainer.fit()

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.2+cu121
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Training NAR Transformer on cuda...


Epoch 1 [Val]: 100%|██████████| 13/13 [00:00<00:00, 19.20it/s, val_loss=0.4537]


Epoch 1 | Train Loss: 0.4496 | Val Loss: 0.4023 | LR: 0.000398 | DTW: 0.0000
Saved best model (Val Loss: 0.4023)


Epoch 2 [Val]: 100%|██████████| 13/13 [00:00<00:00, 33.63it/s, val_loss=0.4334]


Epoch 2 | Train Loss: 0.4315 | Val Loss: 0.3769 | LR: 0.000394 | DTW: 0.0000
Saved best model (Val Loss: 0.3769)


Epoch 3 [Val]:  23%|██▎       | 3/13 [01:12<04:02, 24.28s/it, val_loss=0.2616]


KeyboardInterrupt: 